# KEEPCODING · AI Engineering · M4 OpenClaw

## Clase 5 · Lab 1 · Crea tu primer skill

En este laboratorio escribes, validas y pruebas tu primer **skill** estilo OpenClaw: un fichero `SKILL.md` con frontmatter YAML que enseña a un agente **cuándo y cómo** usar una tool. Todo en local, sin cloud, sin instalar OpenClaw.

El hilo conductor sigue siendo **One Piece**: construiremos un skill `one-piece-bounty` que el agente consulta cuando alguien pregunta por la recompensa de un personaje.

## Para quién es

Alumnado del curso que ya conoce qué es un agente con tools (M1-M3) y quiere entender la capa que falta: cómo se le enseña al modelo cuándo usar una tool. No hace falta instalar OpenClaw para este lab.

## Qué vas a aprender

- Los **3 conceptos** del modelo OpenClaw: **Tool** actúa · **Skill** orienta · **Plugin** distribuye.
- Cómo se estructura un `SKILL.md`: frontmatter YAML (cuándo) + body Markdown (cómo).
- Cómo **validar** un skill: campos obligatorios, formato del frontmatter, dry-run.
- Cómo **probar** el skill alimentándolo a un agente real con LangChain (sin OpenClaw).
- Cómo evolucionar un skill: añadir ejemplos, seguridad, desambiguación.

## Conecta con

- **Slides M4** · sección "Cómo crear un skill · paso a paso".
- **M1 (Agentes con LangChain)** · el agente que consume el skill es el mismo `create_agent` que ya conoces.
- **M5 (Harness Engineering)** · OpenClaw es un harness real · su skill loader es uno de los 6 pilares.
- **Claude Code (clase-1 M4)** · las skills de Claude Code siguen el mismo patrón frontmatter + body.

## Stack

`PyYAML` para validar frontmatter · `langchain>=1.0` + `langchain-openai` para probarlo con un agente.

## Hardware

T4 (Colab Free) sobra · este notebook ni necesita GPU. Todo CPU, modelo vía API.


## Guion del laboratorio

1. **Setup**: instalar PyYAML y configurar `OPENAI_API_KEY`.
2. **Tool vs Skill vs Plugin**: la teoría en 3 líneas.
3. **Anatomía de un skill**: frontmatter + body, con ejemplo.
4. **Escribimos `one-piece-bounty/SKILL.md`**: el skill real, listo para usar.
5. **Validador estático**: comprueba frontmatter, campos obligatorios.
6. **Dry-run**: simulamos qué vería el modelo al cargar el skill.
7. **Probarlo en un agente**: `create_agent` con la tool real, y el skill inyectado en el system prompt.
8. **Iterar**: añadir ejemplos, prioridad, seguridad.
9. **Recap y retos**.

> Idea clave del lab: una **skill no ejecuta nada**. Es texto. Su valor está en que decide *cuándo* y *cómo* el modelo invoca una tool ya existente.


## 1. Setup

### 1.1. Instalar dependencias

Solo necesitamos `PyYAML` para parsear el frontmatter de los skills. Para el paso 7 (probar el skill en un agente) instalamos también `langchain>=1.0` y `langchain-openai`.

> En Colab esto tarda unos segundos. La opción `-q` reduce el ruido.


In [1]:
# Instalación silenciosa
%pip install -q pyyaml "langchain>=1.0" "langchain-openai" "langgraph>=1.0"
print("Dependencias instaladas.")


Note: you may need to restart the kernel to use updated packages.
Dependencias instaladas.


### 1.2. Configurar la API key

`OPENAI_API_KEY` se usará en el paso 7 (probar el skill en un agente real). Si no la tienes, puedes saltarte ese paso y quedarte con la parte de diseño · validación, que es lo importante del lab.


In [2]:
import os
from getpass import getpass


def cargar_api_key() -> None:
    """Carga OPENAI_API_KEY (opcional)."""
    if os.environ.get("OPENAI_API_KEY"):
        print("OPENAI_API_KEY encontrada.")
        return
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or ""
        if os.environ["OPENAI_API_KEY"]:
            print("OPENAI_API_KEY cargada desde secrets de Colab.")
            return
    except Exception:
        pass
    try:
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY (vacío para saltar el paso 7): ")
        if os.environ["OPENAI_API_KEY"]:
            print("OPENAI_API_KEY configurada.")
        else:
            print("Sin API key: el paso 7 se saltará.")
    except Exception:
        print("Sin API key disponible: el paso 7 se saltará.")


cargar_api_key()


OPENAI_API_KEY encontrada.


## 2. Tool vs Skill vs Plugin

La slide M4 lo dice en una línea:

| Capa | Qué hace | Forma |
|---|---|---|
| **Tool** | Actúa (lee fichero, llama API, calcula) | Función Python + schema |
| **Skill** | Orienta (enseña al modelo *cuándo* y *cómo* usar una tool) | `SKILL.md` con frontmatter + body |
| **Plugin** | Distribuye (bundle reutilizable de skills + tools) | Carpeta versionada |

Las tres semánticas son distintas y ninguna reemplaza a otra. Hoy nos centramos en **Skill**: la pieza intermedia que es texto puro y que cambia cómo decide el modelo.

### Por qué las skills importan

Un agente con 30 tools y sin skills termina llamando a la tool equivocada. Una **skill** es la página de manual: cuando el modelo se enfrenta a una pregunta sobre One Piece y ve cargada la skill `one-piece-bounty`, sabe que tiene que llamar a `buscar_personaje`. Sin skill, depende solo del docstring de la tool · que a veces no basta.


## 3. Anatomía de un skill

Un `SKILL.md` tiene dos partes:

1. **Frontmatter YAML** (entre `---`): metadata estructurada. **El sistema** la lee.
2. **Body Markdown**: instrucciones para el modelo. **El modelo** lo lee.

### Frontmatter mínimo

```yaml
---
name: one-piece-bounty
description: Responder preguntas sobre recompensas de personajes de One Piece
version: 1.0.0
when:
  - "pregunta menciona 'recompensa', 'bounty' o el nombre de un personaje de One Piece"
requires_tools:
  - buscar_personaje
---
```

### Body mínimo

```markdown
# Skill · one-piece-bounty

Cuando el usuario pregunte por la recompensa de un personaje, llama a la tool
`buscar_personaje` con el nombre exacto del personaje. Después formatea la
cifra con separadores de miles y la moneda 'berries'.

## Ejemplos

Pregunta: "¿Cuánto vale Luffy?"  -> buscar_personaje("Luffy")
Pregunta: "Dame la bounty de Sanji"  -> buscar_personaje("Sanji")
```

Eso es todo: el frontmatter dice **cuándo** disparar la skill, el body dice **cómo** comportarse. Un skill loader (en OpenClaw, en Claude Code o el que escribiremos en M5) inyecta el body al system prompt cuando detecta que la skill encaja.


## 4. Escribir el skill `one-piece-bounty`

Construimos el `SKILL.md` real del lab. Lo escribimos como un string Python multilinea y lo guardamos al disco para poder validarlo después.

### Decisiones de diseño

- **`when`** explícita: 3 disparadores claros para que el loader no se confunda.
- **`requires_tools`** explícito: si la tool no está disponible, el loader puede avisar.
- **`priority`** declarada: alta porque el dominio es muy concreto.
- **Body con ejemplos** + sección de **seguridad** (qué NO hacer).


In [3]:
from pathlib import Path

SKILL_DIR = Path("skills/one-piece-bounty")
SKILL_DIR.mkdir(parents=True, exist_ok=True)

SKILL_TEXT = '''---
name: one-piece-bounty
description: Responde preguntas sobre la recompensa (bounty) de un personaje de One Piece, llamando a la tool buscar_personaje.
version: 1.0.0
priority: high
when:
  - "el usuario pregunta por la recompensa, bounty o cifra de un personaje de One Piece"
  - "el usuario menciona un nombre canónico (Luffy, Zoro, Nami, Sanji, Robin, ...)"
  - "el usuario pide comparar dos bounties"
requires_tools:
  - buscar_personaje
---

# Skill · one-piece-bounty

Eres un experto entusiasta en One Piece. Cuando esta skill está activa, el
usuario te ha preguntado por la **recompensa** (bounty) de uno o varios
personajes. Sigue este protocolo:

1. **Llama a `buscar_personaje`** con el nombre tal como aparece en la pregunta.
   Si la pregunta menciona varios personajes, hazlo en paralelo.
2. **No respondas de memoria**: las recompensas cambian según el arco. La tool
   tiene la cifra canónica post-Wano.
3. **Formatea la cifra** con separadores de miles y la moneda `berries`.
4. **Cita la tripulación** si la tool la devuelve.
5. **Estilo**: responde en español, breve y sin rayas largas (em dash).

## Ejemplos

- Pregunta: "¿Cuánta es la recompensa de Luffy?"
  Action: buscar_personaje("Luffy") -> formatear y responder.

- Pregunta: "¿Quién tiene más bounty, Zoro o Sanji?"
  Action: buscar_personaje("Zoro") + buscar_personaje("Sanji"),
  comparar las cifras y responder con la diferencia exacta.

- Pregunta: "Háblame de Dragon Ball"
  Action: NO activar esta skill (no es One Piece).

## Seguridad

- NO inventes una cifra si la tool falla: pide disculpas y di que la tool
  no respondió.
- NO interpretes "the strongest" como bounty: aclara que el bounty mide
  la amenaza para el Gobierno Mundial, no la fuerza absoluta.
'''

skill_path = SKILL_DIR / "SKILL.md"
skill_path.write_text(SKILL_TEXT, encoding="utf-8")
print(f"Skill escrita en: {skill_path}")
print(f"Tamaño: {len(SKILL_TEXT)} bytes")


Skill escrita en: skills/one-piece-bounty/SKILL.md
Tamaño: 1750 bytes


## 5. Validador estático del skill

Un skill no compila al cargarse: hay que **validarlo antes**. Escribimos un mini-validador que comprueba:

- El fichero empieza por `---` y tiene frontmatter parseable como YAML.
- Campos obligatorios presentes: `name`, `description`, `version`, `when`, `requires_tools`.
- `when` es una lista no vacía.
- `requires_tools` es una lista no vacía y todos los items son strings.
- El body tiene un título `# Skill · <name>`.

Esto es exactamente lo que el `skill loader` de un harness (OpenClaw, Claude Code, M5) hace en producción antes de inyectar la skill al system prompt.


In [4]:
import re
import yaml


def parsear_skill(texto: str) -> tuple[dict, str]:
    """Separa frontmatter YAML del body. Devuelve (metadata_dict, body_md)."""
    if not texto.startswith("---\n"):
        raise ValueError("El skill no empieza con frontmatter '---'")
    partes = texto.split("---\n", 2)
    if len(partes) < 3:
        raise ValueError("El frontmatter no está cerrado: faltan los '---' de cierre")
    fm = yaml.safe_load(partes[1]) or {}
    body = partes[2]
    return fm, body


REQUIRED_FIELDS = {"name": str, "description": str, "version": str,
                   "when": list, "requires_tools": list}


def validar_skill(texto: str) -> list[str]:
    """Devuelve lista de errores; vacía si el skill es válido."""
    errores: list[str] = []
    try:
        fm, body = parsear_skill(texto)
    except Exception as e:
        return [f"frontmatter no parseable: {e}"]

    for campo, tipo in REQUIRED_FIELDS.items():
        if campo not in fm:
            errores.append(f"falta campo obligatorio: '{campo}'")
            continue
        if not isinstance(fm[campo], tipo):
            errores.append(f"'{campo}' debe ser {tipo.__name__}, es {type(fm[campo]).__name__}")

    if isinstance(fm.get("when"), list) and not fm["when"]:
        errores.append("'when' no puede estar vacío")
    if isinstance(fm.get("requires_tools"), list):
        if not fm["requires_tools"]:
            errores.append("'requires_tools' no puede estar vacío")
        for t in fm["requires_tools"]:
            if not isinstance(t, str):
                errores.append(f"requires_tools tiene un item no-string: {t!r}")

    if "name" in fm:
        esperado = f"# Skill · {fm['name']}"
        if esperado not in body:
            errores.append(f"el body no contiene el título '{esperado}'")

    return errores


texto = skill_path.read_text(encoding="utf-8")
errores = validar_skill(texto)
if errores:
    print("Skill INVÁLIDA. Errores:")
    for e in errores:
        print(f"  - {e}")
else:
    print("Skill VÁLIDA.")


Skill VÁLIDA.


## 6. Dry-run · ver qué le llegaría al modelo

El skill loader inyecta el **body** (no el frontmatter) al system prompt cuando la skill encaja. Aquí simulamos el output: lo que vería el modelo en su contexto.

Esto es útil para depurar antes de gastar tokens en una llamada real.


In [5]:
fm, body = parsear_skill(texto)
print("=== Frontmatter (lo lee el loader, NO el modelo) ===")
print(yaml.safe_dump(fm, allow_unicode=True, sort_keys=False))
print()
print("=== Body (lo lee el modelo si la skill se activa) ===")
print(body[:600] + ("..." if len(body) > 600 else ""))


=== Frontmatter (lo lee el loader, NO el modelo) ===
name: one-piece-bounty
description: Responde preguntas sobre la recompensa (bounty) de un personaje de One
  Piece, llamando a la tool buscar_personaje.
version: 1.0.0
priority: high
when:
- el usuario pregunta por la recompensa, bounty o cifra de un personaje de One Piece
- el usuario menciona un nombre canónico (Luffy, Zoro, Nami, Sanji, Robin, ...)
- el usuario pide comparar dos bounties
requires_tools:
- buscar_personaje


=== Body (lo lee el modelo si la skill se activa) ===

# Skill · one-piece-bounty

Eres un experto entusiasta en One Piece. Cuando esta skill está activa, el
usuario te ha preguntado por la **recompensa** (bounty) de uno o varios
personajes. Sigue este protocolo:

1. **Llama a `buscar_personaje`** con el nombre tal como aparece en la pregunta.
   Si la pregunta menciona varios personajes, hazlo en paralelo.
2. **No respondas de memoria**: las recompensas cambian según el arco. La tool
   tiene la cifra canónica

## 7. Probar el skill en un agente real

Validar es bueno; ver al agente usarlo es mejor. Montamos un agente con `create_agent` (LangChain v1), una tool real `buscar_personaje` (con fallback de One Piece) y el body del skill como **system prompt**. Si la skill está bien escrita, el agente decidirá llamar a la tool en cuanto vea una pregunta sobre bounties.

> Esta celda necesita `OPENAI_API_KEY`. Si la dejaste vacía en el paso 1.2, sáltatela.


In [6]:
# Tool real con fallback (mismo patrón que M1-M3-M5)
import urllib.request
import json as _json
import unicodedata
from langchain_core.tools import tool

RECOMPENSAS = {
    "luffy": ("Monkey D. Luffy", 3_000_000_000, "Sombrero de Paja"),
    "zoro": ("Roronoa Zoro", 1_111_000_000, "Sombrero de Paja"),
    "nami": ("Nami", 366_000_000, "Sombrero de Paja"),
    "sanji": ("Vinsmoke Sanji", 1_032_000_000, "Sombrero de Paja"),
    "robin": ("Nico Robin", 930_000_000, "Sombrero de Paja"),
    "jinbe": ("Jinbe", 1_100_000_000, "Sombrero de Paja"),
}


def _norm(t: str) -> str:
    """Normaliza el nombre y se queda con la última palabra: "Monkey D. Luffy" -> "luffy"."""
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode()
    palabras = "".join(c if c.isalnum() else " " for c in t.lower()).split()
    return palabras[-1] if palabras else ""


@tool
def buscar_personaje(nombre: str) -> dict:
    """Devuelve la recompensa (bounty) post-Wano de un personaje de One Piece.

    Args:
        nombre: nombre del personaje, e.g. "Luffy" o "Monkey D. Luffy".
    """
    key = _norm(nombre)
    if key in RECOMPENSAS:
        nombre_full, bounty, crew = RECOMPENSAS[key]
        return {"nombre": nombre_full, "recompensa": f"{bounty:,} berries".replace(",", "."), "tripulación": crew}
    return {"error": f"personaje no encontrado: {nombre}"}


# Smoke test
print(buscar_personaje.invoke({"nombre": "Luffy"}))


{'nombre': 'Monkey D. Luffy', 'recompensa': '3.000.000.000 berries', 'tripulación': 'Sombrero de Paja'}


In [7]:
# Inyectar la skill como system_prompt y construir el agente
if os.environ.get("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI
    from langchain.agents import create_agent

    modelo = ChatOpenAI(model="gpt-5-mini")
    agente = create_agent(
        model=modelo,
        tools=[buscar_personaje],
        system_prompt=body,
    )

    pregunta = "¿Quién tiene más recompensa, Zoro o Sanji? Dame las cifras exactas."
    resultado = agente.invoke({"messages": [{"role": "user", "content": pregunta}]})
    print("Pregunta:", pregunta)
    print()
    print("Respuesta del agente:")
    print(resultado["messages"][-1].content)
else:
    print("Sin OPENAI_API_KEY: salto el paso 7. Para probarlo:")
    print("1. añade tu clave al entorno: export OPENAI_API_KEY=...")
    print("2. relanza esta celda.")


Pregunta: ¿Quién tiene más recompensa, Zoro o Sanji? Dame las cifras exactas.

Respuesta del agente:
Roronoa Zoro: 1.111.000.000 berries (tripulación: Sombrero de Paja).
Vinsmoke Sanji: 1.032.000.000 berries (tripulación: Sombrero de Paja).
Zoro tiene más recompensa; la diferencia es de 79.000.000 berries.


## 8. Iterar el skill · evolucionar el body

Un skill no se escribe del tirón: se itera cuando observas comportamientos raros del agente. Patrones típicos de iteración:

- **Falsos positivos** (la skill se activa cuando no toca): refinar `when` con más concreción o añadir un `not_when`.
- **Falsos negativos** (la skill no se activa cuando debería): añadir más sinónimos a `when`.
- **Output inconsistente**: añadir un ejemplo más al body, sección "Format".
- **Llamada a tool con args mal**: añadir ejemplo explícito tool call con args.

### Ejercicio de iteración

Edita el `SKILL.md` para que:

1. La skill **también** se active si el usuario menciona "los Sombrero de Paja" como grupo.
2. Cuando la pregunta sea sobre "el más buscado de la tripulación", el agente debe consultar los 10 miembros y devolver el top-1.

Pista: añade un punto a `when` y una sección "Si la pregunta es sobre el grupo entero..." al body.


## 9. Recap y retos

### Qué has construido

- Un `SKILL.md` **completo** para el dominio One Piece, con frontmatter + body + ejemplos.
- Un **validador estático** que comprueba campos obligatorios.
- Un **dry-run** que muestra qué vería el modelo al cargar la skill.
- (Opcional) Un **agente real** con `create_agent` que consume tu skill como system prompt.

### Ideas clave

- Una skill **no ejecuta nada**. Es texto que decide *cuándo* y *cómo* invocar tools.
- El **frontmatter** lo lee el sistema (loader); el **body** lo lee el modelo.
- Las skills son **versionables en git**, **portables** entre harnesses y **iterables** sin tocar código.
- El patrón `frontmatter + body` aparece en OpenClaw, Claude Code, Cursor y en general en cualquier harness moderno: aprenderlo aquí te sirve para todos.

### Retos para practicar

1. **Crea un segundo skill**: `wanted-poster-format`, que enseñe al agente a presentar la respuesta como un cartel de "Se busca" con cifra en grande. Comprueba que las 2 skills coexisten sin chocar.
2. **Loader real**: escribe una función `cargar_skills(carpeta)` que recorra `skills/`, valide cada `SKILL.md` y devuelva un dict `{name: body}` listo para inyectar al system prompt.
3. **`when` con embeddings**: piensa cómo escalarías esto cuando tienes 50 skills. Pista: ranking por similitud entre la pregunta del usuario y la `description` de cada skill.
4. **Plugin**: empaqueta `one-piece-bounty/` en un `plugin.json` con metadata para distribuirlo.

### Conecta con

- **M5** (Harness Engineering): el skill loader es uno de los 6 pilares. En ese lab montas el loader desde cero.
- **M6** (n8n): un nodo n8n puede invocar tu skill como prompt template.
- **Clase 1 M4 (skills)**: el patrón de Claude Code es el mismo frontmatter + body que acabas de usar.
